# Data Prep

In [1]:
import zipfile
from pathlib import Path
import rasterio
from rasterio.merge import merge
import shutil

## Merging zipfiled tifs

In [30]:

country = "dnk"
year = 2022
dataset = "bare_before"
folder = "Bare"

# Use Linux paths when running Python in WSL
input_dir = f"/home/georg/data/LEON_P5_BII/EO_data_raw/{folder}/{dataset}_{country}_{year}"
output_dir = f"/home/georg/data/LEON_P5_BII/EO_data_prep/{folder}"

# Check if input directory exists
input_path = Path(input_dir)
print(f"Input directory: {input_path}")
print(f"Directory exists: {input_path.exists()}")

# Get all zip files
zip_files = list(input_path.glob("*.zip"))
print(f"Found {len(zip_files)} zip files")

if not zip_files:
    print("ERROR: No zip files found!")
    raise FileNotFoundError(f"No zip files found in {input_dir}")

# Create temp directory
Path("temp").mkdir(exist_ok=True)

# Extract and merge tif files
tif_files = []
for zf in zip_files:
    print(f"Processing: {zf.name}")
    with zipfile.ZipFile(zf) as z:
        # Find .tif file in zip
        tif_matches = [n for n in z.namelist() if n.endswith('.tif')]
        if not tif_matches:
            print(f"  WARNING: No .tif files in {zf.name}")
            continue
        tif_name = tif_matches[0]
        z.extract(tif_name, "temp")
        tif_files.append(f"temp/{tif_name}")

print(f"\nTotal .tif files extracted: {len(tif_files)}")

# Open all tif files
src_files = [rasterio.open(f) for f in tif_files]

# Merge them
mosaic, out_transform = merge(src_files)

# Save merged result
Path(output_dir).mkdir(parents=True, exist_ok=True)
with rasterio.open(
    f"{output_dir}/{dataset}_{country}_{year}.tif", "w",
    driver="GTiff",
    height=mosaic.shape[1],
    width=mosaic.shape[2],
    count=mosaic.shape[0],
    dtype=mosaic.dtype,
    transform=out_transform,
    crs=src_files[0].crs,
    compress='LZW',           # LZW compression
    # predictor=2,              # Improves compression for continuous data not categorical (assume similar neighbors)
    tiled=True,               # Better performance with compression
    blockxsize=256,           # Tile size
    blockysize=256
) as dest:
    dest.write(mosaic)

print(f"Merged file saved to: {output_dir}/{dataset}_{country}_{year}.tif")

# Cleanup
for src in src_files:
    src.close()

shutil.rmtree("temp")

Input directory: /home/georg/data/LEON_P5_BII/EO_data_raw/Bare/bare_before_dnk_2022
Directory exists: True
Found 24 zip files
Processing: CLMS_HRLVLCC_CPBSB_S2022_R10m_E43N34_03035_V01_R00.zip
Processing: CLMS_HRLVLCC_CPBSB_S2022_R10m_E43N36_03035_V01_R00.zip
Processing: CLMS_HRLVLCC_CPBSB_S2022_R10m_E45N36_03035_V01_R00.zip
Processing: CLMS_HRLVLCC_CPBSB_S2022_R10m_E44N37_03035_V01_R00.zip
Processing: CLMS_HRLVLCC_CPBSB_S2022_R10m_E42N35_03035_V01_R00.zip
Processing: CLMS_HRLVLCC_CPBSB_S2022_R10m_E46N38_03035_V01_R00.zip
Processing: CLMS_HRLVLCC_CPBSB_S2022_R10m_E44N38_03035_V01_R00.zip
Processing: CLMS_HRLVLCC_CPBSB_S2022_R10m_E45N37_03035_V01_R00.zip
Processing: CLMS_HRLVLCC_CPBSB_S2022_R10m_E45N38_03035_V01_R00.zip
Processing: CLMS_HRLVLCC_CPBSB_S2022_R10m_E41N34_03035_V01_R00.zip
Processing: CLMS_HRLVLCC_CPBSB_S2022_R10m_E44N35_03035_V01_R00.zip
Processing: CLMS_HRLVLCC_CPBSB_S2022_R10m_E45N35_03035_V01_R00.zip
Processing: CLMS_HRLVLCC_CPBSB_S2022_R10m_E41N36_03035_V01_R00.zip
Pro

## Only merge zipfiles within Country bounding box

In [ ]:
# 2018
import zipfile
import rasterio
from rasterio.warp import transform_bounds
from pathlib import Path
import shutil
import subprocess

# Denmark bounding box in WGS84 (lat/lon)
# bbox_wgs84 = (8.076389, 54.559029, 15.193056, 57.751526) # dnk
bbox_wgs84 = (3.360782, 50.723492, 7.227095, 53.554585) # nld

country = "nld"
year = 2018
dataset = "small_woody_features"
folder = "Small_Woody_Features"

# Input and output directories
input_dir = Path(f"/home/georg/data/LEON_P5_BII/EO_data_raw/Small_Woody_Features/small_woody_features_eu_{year}")
output_dir = Path(f"/home/georg/data/LEON_P5_BII/EO_data_prep/{folder}")
output_dir.mkdir(parents=True, exist_ok=True)

temp_dir = Path("temp")
temp_dir.mkdir(exist_ok=True)

# Find all zip files matching SWF_2021 pattern
zip_files = list(input_dir.glob(f"SWF_{year}_*.zip"))
print(f"Found {len(zip_files)} zip files")

# Get CRS from first tile to transform bbox
sample_zip = zip_files[0]
with zipfile.ZipFile(sample_zip) as z:
    # Find WVM .tif inside nested folders
    tif_name = [n for n in z.namelist() if n.endswith('.tif') and 'WVM_' in n][0]
    z.extract(tif_name, temp_dir)
    sample_tif = temp_dir / tif_name
    with rasterio.open(sample_tif) as src:
        tile_crs = src.crs
shutil.rmtree(temp_dir)

# Transform Denmark bbox to tile CRS
bbox_proj = transform_bounds("EPSG:4326", tile_crs, *bbox_wgs84)
print(f"Denmark bbox in {tile_crs}: {bbox_proj}")

# Extract relevant WVM tiles
temp_dir.mkdir(exist_ok=True)
tif_files = []

for zf in zip_files:
    with zipfile.ZipFile(zf) as z:
        # Look for WVM .tif inside nested folders
        tif_matches = [n for n in z.namelist() if n.endswith('.tif') and 'WVM_' in n]
        if not tif_matches:
            continue
        tif_name = tif_matches[0]
        z.extract(tif_name, temp_dir)
        tif_path = temp_dir / tif_name

        with rasterio.open(tif_path) as src:
            left, bottom, right, top = src.bounds
            # Intersection check in projected CRS
            if right < bbox_proj[0] or left > bbox_proj[2] or top < bbox_proj[1] or bottom > bbox_proj[3]:
                print(f"Skipping tile: {zf.name} (outside Denmark)")
                tif_path.unlink()
            else:
                print(f"Including tile: {zf.name}")
                tif_files.append(str(tif_path))

print(f"\nTotal .tif files selected: {len(tif_files)}")
if not tif_files:
    raise FileNotFoundError("No relevant WVM tiles found for Denmark!")

# Build VRT using GDAL
vrt_file = output_dir / f"{country}_mosaic.vrt"
merged_file = output_dir / f"{dataset}_{country}_{year}_merged.tif"

# gdalbuildvrt command
cmd_vrt = ["gdalbuildvrt", str(vrt_file)] + tif_files
print("Building VRT...")
subprocess.run(cmd_vrt)

# Convert VRT to GeoTIFF with compression
cmd_translate = [
    "gdal_translate", str(vrt_file), str(merged_file),
    "-co", "COMPRESS=LZW", "-co", "TILED=YES"
]
print("Converting VRT to GeoTIFF...")
subprocess.run(cmd_translate)

print(f"Merged file saved to: {merged_file}")

# Cleanup
shutil.rmtree(temp_dir)


Found 282 zip files
Denmark bbox in EPSG:3035: (3852900.770419745, 3071690.8233141205, 4137258.6935715666, 3403126.3672772245)
Skipping tile: SWF_2018_005m_E30N21_03035_V1_0.zip (outside Denmark)
Skipping tile: SWF_2018_005m_E48N19_03035_V1_0.zip (outside Denmark)
Skipping tile: SWF_2018_005m_E50N55_03035_V1_0.zip (outside Denmark)
Skipping tile: SWF_2018_005m_E30N17_03035_V1_0.zip (outside Denmark)
Skipping tile: SWF_2018_005m_E50N39_03035_V1_0.zip (outside Denmark)
Skipping tile: SWF_2018_005m_E48N45_03035_V1_0.zip (outside Denmark)
Skipping tile: SWF_2018_005m_E30N35_03035_V1_0.zip (outside Denmark)
Skipping tile: SWF_2018_005m_E40N27_03035_V1_0.zip (outside Denmark)
Skipping tile: SWF_2018_005m_E44N33_03035_V1_0.zip (outside Denmark)
Skipping tile: SWF_2018_005m_E40N37_03035_V1_0.zip (outside Denmark)
Skipping tile: SWF_2018_005m_E46N31_03035_V1_0.zip (outside Denmark)
Skipping tile: SWF_2018_005m_E18N13_03035_V1_0.zip (outside Denmark)
Skipping tile: SWF_2018_005m_E28N23_03035_V1_

In [17]:
#2021
import zipfile
import rasterio
from rasterio.warp import transform_bounds
from pathlib import Path
import shutil
import subprocess

# Bounding box (choose country)
bbox_wgs84 = (8.076389, 54.559029, 15.193056, 57.751526)  # Denmark
# bbox_wgs84 = (3.360782, 50.723492, 7.227095, 53.554585)  # Netherlands

country = "dnk"
year = 2021
dataset = "small_woody_features"
folder = "Small_Woody_Features"

# Input and output directories
input_dir = Path(f"/home/georg/data/LEON_P5_BII/EO_data_raw/Small_Woody_Features/small_woody_features_eu_{year}")
output_dir = Path(f"/home/georg/data/LEON_P5_BII/EO_data_prep/{folder}")
output_dir.mkdir(parents=True, exist_ok=True)

temp_dir = Path("temp")
temp_dir.mkdir(exist_ok=True)

# Find all zip files matching SWF_S2021 pattern
zip_files = list(input_dir.glob(f"SWF_S{year}_*.zip"))
print(f"Found {len(zip_files)} zip files")

if not zip_files:
    raise FileNotFoundError("No zip files found! Check directory or pattern.")

# --- Get CRS from first tile to transform bbox (use CLMS_HRLSLF_SWF_ tifs) ---
sample_zip = zip_files[0]
with zipfile.ZipFile(sample_zip) as z:
    # Find the CLMS/HRLSLF TIFF inside nested folders
    tif_matches = [n for n in z.namelist() if n.endswith(".tif") and "CLMS_HRLSLF_SWF_" in n]
    if not tif_matches:
        # Fallback: print a few names to help debug if structure differs
        print(f"No CLMS_HRLSLF_SWF_ .tif in {sample_zip.name}. First entries:", z.namelist()[:10])
        raise FileNotFoundError("Zip structure unexpected: no CLMS_HRLSLF_SWF_ TIFF found.")
    tif_name = tif_matches[0]
    z.extract(tif_name, temp_dir)
    sample_tif = temp_dir / tif_name
    with rasterio.open(sample_tif) as src:
        tile_crs = src.crs
shutil.rmtree(temp_dir)  # clean and recreate after CRS check

# Transform bbox to tile CRS
bbox_proj = transform_bounds("EPSG:4326", tile_crs, *bbox_wgs84)
print(f"{country.upper()} bbox in {tile_crs}: {bbox_proj}")

# --- Extract relevant CLMS/HRLSLF SWF tiles ---
temp_dir.mkdir(exist_ok=True)
tif_files = []

for zf in zip_files:
    # Optional early filter by country code in the filename (keeps it fast)
    # Comment out if you want bbox-only filtering
    # if "_DK_" not in zf.name:
    #     continue

    with zipfile.ZipFile(zf) as z:
        tif_matches = [n for n in z.namelist() if n.endswith(".tif") and "CLMS_HRLSLF_SWF_" in n]
        if not tif_matches:
            print(f"Skipping {zf.name}: No CLMS_HRLSLF_SWF_ .tif found")
            continue

        tif_name = tif_matches[0]
        z.extract(tif_name, temp_dir)
        tif_path = temp_dir / tif_name

        # Intersection check in the tile CRS
        with rasterio.open(tif_path) as src:
            left, bottom, right, top = src.bounds
            if right < bbox_proj[0] or left > bbox_proj[2] or top < bbox_proj[1] or bottom > bbox_proj[3]:
                print(f"Skipping tile: {zf.name} (outside {country.upper()})")
                tif_path.unlink(missing_ok=True)
            else:
                print(f"Including tile: {zf.name}")
                tif_files.append(str(tif_path))

print(f"\nTotal .tif files selected: {len(tif_files)}")
if not tif_files:
    raise FileNotFoundError(f"No relevant CLMS_HRLSLF_SWF TIFFs found for {country.upper()}!")

# --- Build VRT using GDAL (memory-efficient) ---
vrt_file = output_dir / f"{country}_mosaic.vrt"
merged_file = output_dir / f"{dataset}_{country}_{year}_merged.tif"

cmd_vrt = ["gdalbuildvrt", str(vrt_file)] + tif_files
print("Building VRT...")
subprocess.run(cmd_vrt, check=True)

# Convert VRT to GeoTIFF with compression
cmd_translate = [
    "gdal_translate", str(vrt_file), str(merged_file),
    "-co", "COMPRESS=LZW", "-co", "TILED=YES"
]
print("Converting VRT to GeoTIFF...")
subprocess.run(cmd_translate, check=True)

print(f"Merged file saved to: {merged_file}")

# Cleanup
shutil.rmtree(temp_dir)


Found 500 zip files
DNK bbox in EPSG:3035: (4196533.820971617, 3494767.655439731, 4656723.776447127, 3861204.2421139604)
Skipping tile: SWF_S2021_005m_E44N43_NO_SE_03035_V01_R01.zip (outside DNK)
Skipping tile: SWF_S2021_005m_E34N30_FR_03035_V01_R01.zip (outside DNK)
Skipping tile: SWF_S2021_005m_E41N23_FR_IT_03035_V01_R01.zip (outside DNK)
Skipping tile: SWF_S2021_005m_E42N40_NO_03035_V01_R01.zip (outside DNK)
Skipping tile: SWF_S2021_005m_E37N20_ES_03035_V01_R01.zip (outside DNK)
Skipping tile: SWF_S2021_005m_E45N49_NO_SE_03035_V01_R01.zip (outside DNK)
Skipping tile: SWF_S2021_005m_E45N24_HR_IT_03035_V01_R01.zip (outside DNK)
Skipping tile: SWF_S2021_005m_E47N26_AT_HR_HU_SI_03035_V01_R01.zip (outside DNK)
Skipping tile: SWF_S2021_005m_E45N25_IT_SI_03035_V01_R01.zip (outside DNK)
Skipping tile: SWF_S2021_005m_E44N23_IT_03035_V01_R01.zip (outside DNK)
Skipping tile: SWF_S2021_005m_E40N39_NO_03035_V01_R01.zip (outside DNK)
Skipping tile: SWF_S2021_005m_E48N39_SE_03035_V01_R01.zip (outs

### Check directory

In [9]:
# Check directory
ghsl_base = Path("/home/georg/data/LEON_P5_BII/EO_data_raw/Small_Woody_Features/small_woody_features_eu_2021")

print(f"GHSL directory exists: {ghsl_base.exists()}")
print(f"\nContents of {ghsl_base}:")
print("="*60)

if ghsl_base.exists():
    # Show subdirectories and their contents
    for item in sorted(ghsl_base.iterdir()):
        if item.is_dir():
            print(f"\n📁 {item.name}/")
            # Show what's inside each subdirectory (one level deep)
            for subitem in sorted(item.iterdir())[:10]:  # Limit to first 10 items
                if subitem.is_dir():
                    print(f"   📁 {subitem.name}/")
                else:
                    print(f"   📄 {subitem.name}")
            if len(list(item.iterdir())) > 10:
                print(f"   ... and {len(list(item.iterdir())) - 10} more items")
        else:
            print(f"📄 {item.name}")
else:
    print("Directory does not exist!")

GHSL directory exists: True

Contents of /home/georg/data/LEON_P5_BII/EO_data_raw/Small_Woody_Features/small_woody_features_eu_2021:
📄 SWF_S2021_005m_E09N27_PT_03035_V01_R01.zip
📄 SWF_S2021_005m_E10N25_PT_03035_V01_R01.zip
📄 SWF_S2021_005m_E11N25_PT_03035_V01_R01.zip
📄 SWF_S2021_005m_E12N22_PT_03035_V01_R01.zip
📄 SWF_S2021_005m_E12N23_PT_03035_V01_R01.zip
📄 SWF_S2021_005m_E12N25_PT_03035_V01_R01.zip
📄 SWF_S2021_005m_E13N23_PT_03035_V01_R01.zip
📄 SWF_S2021_005m_E15N10_ES_03035_V01_R01.zip
📄 SWF_S2021_005m_E15N11_ES_03035_V01_R01.zip
📄 SWF_S2021_005m_E16N10_ES_03035_V01_R01.zip
📄 SWF_S2021_005m_E16N11_ES_03035_V01_R01.zip
📄 SWF_S2021_005m_E17N09_ES_03035_V01_R01.zip
📄 SWF_S2021_005m_E17N10_ES_03035_V01_R01.zip
📄 SWF_S2021_005m_E17N15_PT_03035_V01_R01.zip
📄 SWF_S2021_005m_E18N09_ES_03035_V01_R01.zip
📄 SWF_S2021_005m_E18N12_PT_03035_V01_R01.zip
📄 SWF_S2021_005m_E18N14_%20PT_03035_V01_R01.zip
📄 SWF_S2021_005m_E18N15_PT_03035_V01_R01.zip
📄 SWF_S2021_005m_E19N09_ES_03035_V01_R01.zip
📄 SWF_S20

## Merging multiple countries and years in folder (not zipped)

In [3]:
# Merging multiple countries and years in folder (not zipped)

import rasterio
from rasterio.merge import merge
from pathlib import Path

dataset = "ghsl_pop"
output_base = "/home/georg/data/LEON_P5_BII/EO_data_prep/GHSL"

# Define what to process
countries_years = {
    "ndl": [2015, 2020, 2025],
    "cnk": [2015, 2020, 2025],
    "uk": [2015, 2025]
}


for country, years in countries_years.items():
    for year in years:
        print(f"\n{'='*60}")
        print(f"Processing: {country.upper()} - {year}")
        print(f"{'='*60}")
        
        # Adjust base directory structure based on your actual folder organization
        base_dir = f"/home/georg/data/LEON_P5_BII/EO_data_raw/GHSL/{year}"
        
        input_path = Path(base_dir)
        if not input_path.exists():
            print(f"⚠ Directory not found: {base_dir}")
            continue
        
        # Find all .tif files
        tif_files = list(input_path.rglob("*.tif"))
        print(f"Found {len(tif_files)} .tif files")
        
        if not tif_files:
            print(f"⚠ No .tif files found for {country} {year}")
            continue
        
        # Open, merge, and save
        try:
            src_files = [rasterio.open(str(f)) for f in tif_files]
            mosaic, out_transform = merge(src_files)
            
            output_dir = Path(output_base)
            output_dir.mkdir(parents=True, exist_ok=True)
            output_file = output_dir / f"{dataset}_{country}_{year}_merged.tif"
            
            with rasterio.open(
                output_file, "w",
                driver="GTiff",
                height=mosaic.shape[1],
                width=mosaic.shape[2],
                count=mosaic.shape[0],
                dtype=mosaic.dtype,
                transform=out_transform,
                crs=src_files[0].crs,
                compress='LZW',
                tiled=True,
                blockxsize=256,
                blockysize=256
            ) as dest:
                dest.write(mosaic)
            
            # Cleanup
            for src in src_files:
                src.close()
            
            print(f"✓ Saved: {output_file}")
            
        except Exception as e:
            print(f"✗ Error processing {country} {year}: {e}")


Processing: NDL - 2015
Found 4 .tif files
✓ Saved: /home/georg/data/LEON_P5_BII/EO_data_prep/GHSL/ghsl_pop_ndl_2015_merged.tif

Processing: NDL - 2020
Found 4 .tif files
✓ Saved: /home/georg/data/LEON_P5_BII/EO_data_prep/GHSL/ghsl_pop_ndl_2020_merged.tif

Processing: NDL - 2025
Found 4 .tif files
✓ Saved: /home/georg/data/LEON_P5_BII/EO_data_prep/GHSL/ghsl_pop_ndl_2025_merged.tif

Processing: CNK - 2015
Found 4 .tif files
✓ Saved: /home/georg/data/LEON_P5_BII/EO_data_prep/GHSL/ghsl_pop_cnk_2015_merged.tif

Processing: CNK - 2020
Found 4 .tif files
✓ Saved: /home/georg/data/LEON_P5_BII/EO_data_prep/GHSL/ghsl_pop_cnk_2020_merged.tif

Processing: CNK - 2025
Found 4 .tif files
✓ Saved: /home/georg/data/LEON_P5_BII/EO_data_prep/GHSL/ghsl_pop_cnk_2025_merged.tif

Processing: UK - 2015
Found 4 .tif files
✓ Saved: /home/georg/data/LEON_P5_BII/EO_data_prep/GHSL/ghsl_pop_uk_2015_merged.tif

Processing: UK - 2025
Found 4 .tif files
✓ Saved: /home/georg/data/LEON_P5_BII/EO_data_prep/GHSL/ghsl_pop

In [ ]:
dataset = "ghsl_pop"
output_base = "/home/georg/data/LEON_P5_BII/EO_data_prep/GHSL"

# Clip to Country boundaries

In [3]:
import geopandas as gpd
from pathlib import Path

# Directory with gpkg files
gpkg_dir = Path("/home/georg/data/LEON_P5_BII/Country_Boundaries")

# Load all gpkg files
uk = gpd.read_file(gpkg_dir / "gadm41_GBR_adm2_filtered4_3035.gpkg")
dnk = gpd.read_file(gpkg_dir / "gadm41_dnk_adm_0_3035.gpkg")
nld = gpd.read_file(gpkg_dir / "gadm41_nld_adm_0_3035.gpkg")

In [ ]:
from rasterio.mask import mask
from pathlib import Path

### For folder ###

## Config ##
aoi = uk # Using country boundaries for clipping

country = "nld"
year = 2023
dataset = "bare_before"
folder = "Bare"


# Find all .tif files in subfolders
tif_dir = Path(f"/home/georg/data/LEON_P5_BII/EO_data_prep/{folder}/{dataset}_{country}_{year}_merged.tif")
tif_files = list(tif_dir.rglob("*.tif"))

print(f"Found {len(tif_files)} .tif files")

# Load and clip each tif with each country boundary
for tif_file in tif_files:
    print(f"\nProcessing: {tif_file.name}")
    
    with rasterio.open(tif_file) as src:
        # Clip with GBR
        aoi_geom = [aoi.geometry.unary_union]
        aoi_clipped, aoi_transform = mask(src, aoi_geom, crop=True)
        
        # Save clipped result
        output_path = f"/home/georg/data/LEON_P5_BII/EO_data_prep/{folder}/{dataset}_{country}_{year}.tif"
        with rasterio.open(output_path, 'w',
                          driver='GTiff',
                          height=aoi_clipped.shape[1],
                          width=aoi_clipped.shape[2],
                          count=src.count,
                          dtype=aoi_clipped.dtype,
                          crs=src.crs,
                          transform=aoi_transform,
                          compress='LZW') as dst:
            dst.write(aoi_clipped)
        
        print(f"Saved: {output_path}")

Found 9 .tif files

Processing: canopy_height_netherlands_2m-0000000000-0000000000.tif


NameError: name 'rasterio' is not defined

In [ ]:
import rasterio
from rasterio.mask import mask
from pathlib import Path

### For files ###

## Config ##
aoi = nld # Using country boundaries for clipping

country = "nld"
year = 2023
dataset = "bare_after"
folder = "Bare" 

# Path can be a file or folder
tif_path = Path(f"/home/georg/data/LEON_P5_BII/EO_data_prep/{folder}/{dataset}_{country}_{year}.tif")

# Check if it's a file or folder
if tif_path.is_file():
    tif_files = [tif_path]
elif tif_path.is_dir():
    tif_files = list(tif_path.rglob("*.tif"))
else:
    raise FileNotFoundError(f"Path not found: {tif_path}")

print(f"Found {len(tif_files)} .tif file(s)")

# Load and clip each tif
for tif_file in tif_files:
    print(f"\nProcessing: {tif_file.name}")
    
    with rasterio.open(tif_file) as src:
        # Clip with AOI
        aoi_geom = [aoi.geometry.unary_union]
        aoi_clipped, aoi_transform = mask(src, aoi_geom, crop=True)
        
        # Generate output filename
        output_name = tif_file.stem + "_clip.tif"
        output_path = tif_file.parent / output_name
        
        # Save clipped result
        with rasterio.open(output_path, 'w',
                          driver='GTiff',
                          height=aoi_clipped.shape[1],
                          width=aoi_clipped.shape[2],
                          count=src.count,
                          dtype=aoi_clipped.dtype,
                          crs=src.crs,
                          transform=aoi_transform,
                          compress='LZW') as dst:
            dst.write(aoi_clipped)
        
        print(f"Saved: {output_path}")

AOI already in EPSG:3035
Clipping c_gls_LSP300-LENGTH-S1_202201010000_GLOBE_OLCI_V2.0.2.tiff using AOI from memory...
Creating output file that is 2390P x 1072L.
Processing /home/georg/data/LEON_P5_BII/EO_data_raw/Phenology/c_gls_LSP300-LENGTH-S1_202201010000_GLOBE_OLCI_V2.0.2_cog/c_gls_LSP300-LENGTH-S1_202201010000_GLOBE_OLCI_V2.0.2_cog/c_gls_LSP300-LENGTH-S1_202201010000_GLOBE_OLCI_V2.0.2.tiff [1/1] : 0Using internal nodata values (e.g. -9999) for image /home/georg/data/LEON_P5_BII/EO_data_raw/Phenology/c_gls_LSP300-LENGTH-S1_202201010000_GLOBE_OLCI_V2.0.2_cog/c_gls_LSP300-LENGTH-S1_202201010000_GLOBE_OLCI_V2.0.2_cog/c_gls_LSP300-LENGTH-S1_202201010000_GLOBE_OLCI_V2.0.2.tiff.
...10...20...30...40...50...60...70...80...90...100 - done.
Clipped raster saved to: /home/georg/data/LEON_P5_BII/EO_data_prep/Phenology/LSP300_LENGTH_S1_dnk_2022_clip.tif
Output CRS: EPSG:4326
Output shape: (1072, 2390)
Output bounds: BoundingBox(left=8.078870552260819, bottom=54.56101170123825, right=15.191965

## For CLMS Phenology 

In [20]:
# Multiprocess clipping with GDAL
import subprocess
from pathlib import Path
import tempfile
import rasterio

# --- Config ---
aoi = nld  # Already loaded GeoDataFrame
country = "nld"
year = 2022
dataset = "LSP300_LENGTH_S1"
folder = "c_gls_LSP300-LENGTH-S1_202201010000_GLOBE_OLCI_V2.0.2_cog/c_gls_LSP300-LENGTH-S1_202201010000_GLOBE_OLCI_V2.0.2_cog"

# Input raster
tif_path = Path(f"/home/georg/data/LEON_P5_BII/EO_data_raw/Phenology/{folder}/c_gls_LSP300-LENGTH-S1_202201010000_GLOBE_OLCI_V2.0.2.tiff")
output_path = Path(f"/home/georg/data/LEON_P5_BII/EO_data_prep/Phenology/{dataset}_{country}_{year}_clip.tif")

# Check input file exists
if not tif_path.exists():
    raise FileNotFoundError(f"Input raster not found: {tif_path}")

# --- Ensure AOI is in EPSG:3035 ---
target_crs = "EPSG:3035"
if aoi.crs != target_crs:
    print(f"Reprojecting AOI from {aoi.crs} to {target_crs}")
    aoi = aoi.to_crs(target_crs)
else:
    print(f"AOI already in {target_crs}")

# --- Save AOI to temporary GPKG for GDAL ---
with tempfile.TemporaryDirectory() as tmpdir:
    tmp_gpkg = Path(tmpdir) / "aoi.gpkg"
    aoi.to_file(tmp_gpkg, driver="GPKG")

    # --- GDAL command with reprojection ---
    cmd = [
        "gdalwarp",
        "-cutline", str(tmp_gpkg),
        "-crop_to_cutline",
        "-t_srs", target_crs,
        "-tr", "300", "300",
        "-r", "bilinear",                # ✨ Bilinear for continuous data
        "-of", "GTiff",
        "-co", "COMPRESS=LZW",
        "-co", "TILED=YES",
        "-multi",
        "-wo", "NUM_THREADS=ALL_CPUS",
        str(tif_path),
        str(output_path)
    ]

    print(f"Clipping and reprojecting {tif_path.name}...")
    subprocess.run(cmd, check=True)

print(f"✅ Clipped raster saved to: {output_path}")

# --- Verify output CRS ---
with rasterio.open(output_path) as src:
    print(f"Output CRS: {src.crs}")
    print(f"Output shape: {src.shape}")
    print(f"Output bounds: {src.bounds}")
    print(f"Output resolution: {src.res}")

AOI already in EPSG:3035
Clipping and reprojecting c_gls_LSP300-LENGTH-S1_202201010000_GLOBE_OLCI_V2.0.2.tiff...
Creating output file that is 921P x 1044L.
Processing /home/georg/data/LEON_P5_BII/EO_data_raw/Phenology/c_gls_LSP300-LENGTH-S1_202201010000_GLOBE_OLCI_V2.0.2_cog/c_gls_LSP300-LENGTH-S1_202201010000_GLOBE_OLCI_V2.0.2_cog/c_gls_LSP300-LENGTH-S1_202201010000_GLOBE_OLCI_V2.0.2.tiff [1/1] : 0Using internal nodata values (e.g. -9999) for image /home/georg/data/LEON_P5_BII/EO_data_raw/Phenology/c_gls_LSP300-LENGTH-S1_202201010000_GLOBE_OLCI_V2.0.2_cog/c_gls_LSP300-LENGTH-S1_202201010000_GLOBE_OLCI_V2.0.2_cog/c_gls_LSP300-LENGTH-S1_202201010000_GLOBE_OLCI_V2.0.2.tiff.
Copying nodata values from source /home/georg/data/LEON_P5_BII/EO_data_raw/Phenology/c_gls_LSP300-LENGTH-S1_202201010000_GLOBE_OLCI_V2.0.2_cog/c_gls_LSP300-LENGTH-S1_202201010000_GLOBE_OLCI_V2.0.2_cog/c_gls_LSP300-LENGTH-S1_202201010000_GLOBE_OLCI_V2.0.2.tiff to destination /home/georg/data/LEON_P5_BII/EO_data_prep/Ph

## Mask based on Raster

In [4]:
# Batch compute

import rasterio
from rasterio.warp import reproject, Resampling
import numpy as np
from pathlib import Path

# Config
nitrogen_path = Path("/home/georg/data/LEON_P5_BII/EO_data_prep/GHSL/ghsl_pop_2015_merged.tif")
lc_dir = Path("/home/georg/data/LEON_P5_BII/EO_data_prep/CLCplus")
output_dir = Path("/home/georg/data/LEON_P5_BII/EO_data_prep/GHSL")

output_dir.mkdir(exist_ok=True, parents=True)

# Process each country
for lc_path in lc_dir.glob("clcplus_*_2023.tif"):
    country = lc_path.stem.split('_')[1]  # dnk, nld, uk
    output_path = output_dir / f"ghsl_{country}_15.tif"
    
    print(f"Processing {country.upper()}...")
    
    with rasterio.open(lc_path) as lc, rasterio.open(nitrogen_path) as src:
        # Get valid LC pixel mask
        lc_data = lc.read(1)
        valid_mask = lc_data != lc.nodata if lc.nodata else np.ones_like(lc_data, dtype=bool)
        
        # Reproject nitrogen to LC grid
        output = np.empty((src.count, lc.height, lc.width), dtype=src.dtypes[0])
        
        reproject(
            source=rasterio.band(src, range(1, src.count + 1)),
            destination=output,
            src_transform=src.transform,
            src_crs=src.crs,
            dst_transform=lc.transform,
            dst_crs=lc.crs,
            resampling=Resampling.bilinear
        )
        
        # Mask invalid LC pixels
        output[:, ~valid_mask] = src.nodata if src.nodata else -9999
        
        # Save
        profile = lc.profile.copy()
        profile.update(dtype=src.dtypes[0], count=src.count, nodata=src.nodata or -9999, compress='LZW')
        
        with rasterio.open(output_path, 'w', **profile) as dst:
            dst.write(output)
    
    print(f"  ✓ {output_path}")

print("\nDone!")

Processing NLD...
  ✓ /home/georg/data/LEON_P5_BII/EO_data_prep/GHSL/ghsl_nld_15.tif
Processing UK...


: 

In [4]:
# Batch compute - preserve GHSL resolution, clip to LC extent

import rasterio
from rasterio.warp import reproject, calculate_default_transform, Resampling
from rasterio.mask import mask
from shapely.geometry import box
import numpy as np
from pathlib import Path


dataset = "ghsl_pop_2025_merged"
year = 2025

# Config
ghsl_path = Path(f"/home/georg/data/LEON_P5_BII/EO_data_prep/GHSL/{dataset}.tif")
lc_dir = Path("/home/georg/data/LEON_P5_BII/EO_data_prep/CLCplus")
output_dir = Path("/home/georg/data/LEON_P5_BII/EO_data_prep/GHSL")

output_dir.mkdir(exist_ok=True, parents=True)

# Process each country
for lc_path in lc_dir.glob("clcplus_*_2023.tif"):
    country = lc_path.stem.split('_')[1]  # dnk, nld, uk
    output_path = output_dir / f"ghsl_{country}_{year}.tif"
    
    print(f"Processing {country.upper()}...")
    
    with rasterio.open(lc_path) as lc, rasterio.open(ghsl_path) as ghsl:
        
        # Get LC bounds in its CRS
        lc_bounds = lc.bounds
        
        # If CRS differs, reproject LC bounds to GHSL CRS
        if lc.crs != ghsl.crs:
            from rasterio.warp import transform_bounds
            ghsl_bounds = transform_bounds(lc.crs, ghsl.crs, *lc_bounds)
        else:
            ghsl_bounds = lc_bounds
        
        # Create bounding box geometry
        bbox = box(*ghsl_bounds)
        
        # Clip GHSL to LC extent (keeps GHSL resolution)
        clipped_data, clipped_transform = mask(
            ghsl, 
            [bbox], 
            crop=True,
            all_touched=True  # Include pixels that touch the boundary
        )
        
        # If CRS differs, reproject the clipped data to LC CRS
        if lc.crs != ghsl.crs:
            print(f"  Reprojecting from {ghsl.crs} to {lc.crs}...")
            
            # Calculate transform for reprojected data (keeps GHSL resolution)
            dst_transform, dst_width, dst_height = calculate_default_transform(
                ghsl.crs, 
                lc.crs, 
                clipped_data.shape[2], 
                clipped_data.shape[1],
                *ghsl_bounds,
                resolution=ghsl.res  # Keep original resolution
            )
            
            # Reproject
            reprojected = np.empty((ghsl.count, dst_height, dst_width), dtype=ghsl.dtypes[0])
            reproject(
                source=clipped_data,
                destination=reprojected,
                src_transform=clipped_transform,
                src_crs=ghsl.crs,
                dst_transform=dst_transform,
                dst_crs=lc.crs,
                resampling=Resampling.bilinear
            )
            
            output_data = reprojected
            output_transform = dst_transform
            output_crs = lc.crs
        else:
            output_data = clipped_data
            output_transform = clipped_transform
            output_crs = ghsl.crs
        
        # Save with GHSL properties but aligned to LC extent
        profile = ghsl.profile.copy()
        profile.update(
            height=output_data.shape[1],
            width=output_data.shape[2],
            transform=output_transform,
            crs=output_crs,
            compress='LZW',
            tiled=True
        )
        
        with rasterio.open(output_path, 'w', **profile) as dst:
            dst.write(output_data)
    
    print(f"  ✓ Resolution: {ghsl.res[0]:.2f}m")
    print(f"  ✓ Shape: {output_data.shape}")
    print(f"  ✓ {output_path}")

print("\nDone!")

Processing NLD...
  Reprojecting from ESRI:54009 to EPSG:3035...
  ✓ Resolution: 100.00m
  ✓ Shape: (1, 3489, 3457)
  ✓ /home/georg/data/LEON_P5_BII/EO_data_prep/GHSL/ghsl_nld_2025.tif
Processing UK...
  Reprojecting from ESRI:54009 to EPSG:3035...
  ✓ Resolution: 100.00m
  ✓ Shape: (1, 4378, 5448)
  ✓ /home/georg/data/LEON_P5_BII/EO_data_prep/GHSL/ghsl_uk_2025.tif
Processing DNK...
  Reprojecting from ESRI:54009 to EPSG:3035...
  ✓ Resolution: 100.00m
  ✓ Shape: (1, 3799, 5121)
  ✓ /home/georg/data/LEON_P5_BII/EO_data_prep/GHSL/ghsl_dnk_2025.tif

Done!


In [ ]:
# Single file compute

import rasterio
from rasterio.warp import reproject, Resampling
import numpy as np

# Paths
nitrogen_path = "/home/georg/data/LEON_P5_BII/EO_data_raw/Nitrogen/nitrogen_soilgrids.tif"
lc_path = "/home/georg/data/LEON_P5_BII/EO_data_prep/CLCplus/clcplus_nld_2023.tif"
output_path = "/home/georg/data/LEON_P5_BII/EO_data_prep/Nitrogen/nitrogen_nld_2023_clipped.tif"

with rasterio.open(lc_path) as lc, rasterio.open(nitrogen_path) as src:
    # Read LC data to get valid pixel mask
    lc_data = lc.read(1)
    valid_mask = lc_data != lc.nodata if lc.nodata else np.ones_like(lc_data, dtype=bool)
    
    # Create output array matching LC dimensions
    output = np.empty((src.count, lc.height, lc.width), dtype=src.dtypes[0])
    
    # Reproject nitrogen to LC grid
    reproject(
        source=rasterio.band(src, range(1, src.count + 1)),
        destination=output,
        src_transform=src.transform,
        src_crs=src.crs,
        dst_transform=lc.transform,
        dst_crs=lc.crs,
        resampling=Resampling.bilinear
    )
    
    # Mask invalid LC pixels
    output[:, ~valid_mask] = src.nodata if src.nodata else -9999
    
    # Save
    profile = lc.profile.copy()
    profile.update(dtype=src.dtypes[0], count=src.count, nodata=src.nodata or -9999, compress='LZW')
    
    with rasterio.open(output_path, 'w', **profile) as dst:
        dst.write(output)

print(f"✓ Saved: {output_path}")